# Embeddings

This notebook generates vector embeddings for every Chunk node in the graph and creates a vector index for semantic search.

**Learning Objectives:**
- Understand embeddings as numerical representations of text meaning
- Generate embeddings using Amazon Nova Multimodal Embeddings
- Store embeddings on Chunk nodes in Neo4j
- Create and query a vector index

Amazon Nova Multimodal Embeddings produces 1024-dimensional vectors (configurable). Two chunks about similar topics will have vectors that point in similar directions, and cosine similarity measures that alignment.

In [ ]:
%pip install "neo4j-graphrag[bedrock] @ git+https://github.com/neo4j-partners/neo4j-graphrag-python.git@bedrock-embeddings" -q

In [ ]:
import os

from lib.data_utils import get_embedder, BedrockConfig
from neo4j_graphrag.indexes import create_vector_index, upsert_vectors
from neo4j import GraphDatabase
from dotenv import load_dotenv

# Load configuration
load_dotenv('../CONFIG.txt')

NEO4J_URI = os.getenv('NEO4J_URI')
NEO4J_USERNAME = os.getenv('NEO4J_USERNAME')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD')

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
driver.verify_connectivity()

embedder = get_embedder()
print(f'Connected to Neo4j!')
print(f'Embedder: {embedder.model_id}')

## Test Embedding Generation

Generate an embedding for a sample sentence to see what the output looks like.

In [ ]:
sample_text = "Apple's iPhone revenue grew 5% year-over-year"
embedding = embedder.embed_query(sample_text)

print(f'Text: "{sample_text}"')
print(f'Dimensions: {len(embedding)}')
print(f'First 5 values: {embedding[:5]}')

## Generate Embeddings for All Chunks

Fetch every Chunk node that does not yet have an embedding, generate one from its text, and store the vector back on the node.

In [ ]:
with driver.session() as session:
    # Fetch chunks without embeddings
    result = session.run("""
        MATCH (c:Chunk)
        WHERE c.embedding IS NULL
        RETURN elementId(c) AS chunk_id, c.text AS text, c.index AS index
        ORDER BY c.index
    """)
    chunks = list(result)

print(f'Found {len(chunks)} chunks without embeddings\n')

# Generate embeddings for each chunk
ids = []
embeddings = []
for chunk in chunks:
    embedding = embedder.embed_query(chunk['text'])
    ids.append(chunk['chunk_id'])
    embeddings.append(embedding)
    print(f'Embedded Chunk {chunk["index"]} ({len(embedding)} dimensions)')

# Batch update all embeddings using the library's upsert function
upsert_vectors(
    driver,
    ids=ids,
    embedding_property='embedding',
    embeddings=embeddings,
)

print(f'\nAll chunks embedded!')

## Create Vector Index

Create a vector index on the `embedding` property of Chunk nodes. This index uses cosine similarity and enables efficient approximate nearest-neighbor search.

In [ ]:
create_vector_index(
    driver,
    name='chunkEmbeddings',
    label='Chunk',
    embedding_property='embedding',
    dimensions=1024,
    similarity_fn='cosine'
)

print('Vector index created!')

# Verify the index
with driver.session() as session:
    result = session.run("""
        SHOW VECTOR INDEXES
        YIELD name, labelsOrTypes, properties, state
        RETURN name, labelsOrTypes, properties, state
    """)
    for record in result:
        print(f'  Index: {record["name"]} on {record["labelsOrTypes"]}.{record["properties"]} ({record["state"]})')

## Test Vector Search

Run a quick search using `VectorRetriever` to verify that semantically similar chunks are returned. The retriever handles embedding the query and searching the index automatically. The next notebook covers retrievers in depth.

In [ ]:
from neo4j_graphrag.retrievers import VectorRetriever

retriever = VectorRetriever(
    driver=driver,
    index_name='chunkEmbeddings',
    embedder=embedder,
    return_properties=['text'],
)

query = "What are Apple's main risk factors?"
print(f'Query: "{query}"\n')
print('=== Vector Search Results ===')

results = retriever.search(query_text=query, top_k=3)
for i, item in enumerate(results.items, 1):
    score = item.metadata.get('score', 0)
    content = item.content if isinstance(item.content, str) else str(item.content)
    print(f'\n{i}. Score: {score:.4f}')
    print(f'   {content[:150]}...')

## Summary

You added semantic search capability to the knowledge graph:

1. **Embeddings** turn text into 1024-dimensional vectors that capture meaning
2. **Vector index** enables fast approximate nearest-neighbor search over those vectors
3. **Cosine similarity** measures how closely a query matches each chunk

The raw vector search shown here works, but it returns only the matched chunks without any graph context. In the next notebook, you will wrap this index in a `VectorRetriever` and connect it to an LLM to build a complete question-answering pipeline.

---

**Next:** [Vector Retriever](03_vector_retriever.ipynb)

In [ ]:
# Cleanup
driver.close()
print('Connection closed.')